In [236]:
import os
import json
from backend.image_processing import *
import geopandas as gpd
from PIL import Image, ImageOps, ImageDraw,ImageFile
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from mpl_toolkits.axes_grid1 import ImageGrid
from datetime import datetime, timedelta
import numpy.ma as ma
from datetime import datetime
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
# import matplotlib.ticker as ticker
# from matplotlib.patches import FancyArrowPatch
# import matplotlib.patheffects as pe

In [237]:
ImageFile.LOAD_TRUNCATED_IMAGES = True

In [238]:
# os.path.basename(file_path[0]).split('_')[3]

In [239]:
phenocam_dir = '/home/ajai-krishna/work/Phenocam_d3/Phenocamdata_local'
data_dir=[os.path.join(phenocam_dir,files) for files in os.listdir(phenocam_dir) if files.endswith('.jpg')]
plot_dir='/home/ajai-krishna/work/Phenocam_d3/Plots'
file_path = [files for files in data_dir]
# file_path=[]
# for data in data_path:
#     if os.path.basename(data).split('_')[3] == '2026':
#         file_path.append(data)
# file_path

In [255]:
df1 = create_image_metadata_df(file_path)
df1

,file_path,image_type,date_time,date,time
0,/home/ajai-krishna/work/Phenocam_d3/Phenocamda...,other,2025_07_28_13_38_47,2025_07_28,1338
1,/home/ajai-krishna/work/Phenocam_d3/Phenocamda...,rgb,2025_07_28_13_38_47,2025_07_28,1338
2,/home/ajai-krishna/work/Phenocam_d3/Phenocamda...,other,2025_07_28_13_40_45,2025_07_28,1340
3,/home/ajai-krishna/work/Phenocam_d3/Phenocamda...,ir,2025_07_28_13_40_45,2025_07_28,1340
4,/home/ajai-krishna/work/Phenocam_d3/Phenocamda...,other,2025_07_28_13_42,2025_07_28,1342
...,...,...,...,...,...
2973,/home/ajai-krishna/work/Phenocam_d3/Phenocamda...,ndvi,2025_08_11_11_30_45,2025_08_11,1130
2974,/home/ajai-krishna/work/Phenocam_d3/Phenocamda...,other,2025_08_12_11_00,2025_08_12,1100
2975,/home/ajai-krishna/work/Phenocam_d3/Phenocamda...,other,2025_08_12_11_00,2025_08_12,1100
2976,/home/ajai-krishna/work/Phenocam_d3/Phenocamda...,other,2025_08_12_11_00,2025_08_12,1100


In [241]:
df_ndvi=df1[df1['image_type']=='NDVI'].reset_index(drop=True)
df_ndvi

,file_path,image_type,date_time,date,time
0,/home/ajai-krishna/work/Phenocam_d3/Phenocamda...,NDVI,2025_07_28_13_42_46,2025_07_28,1342
1,/home/ajai-krishna/work/Phenocam_d3/Phenocamda...,NDVI,2026_01_06_11_31_02,2026_01_06,1131
2,/home/ajai-krishna/work/Phenocam_d3/Phenocamda...,NDVI,2026_01_07_11_01_03,2026_01_07,1101
3,/home/ajai-krishna/work/Phenocam_d3/Phenocamda...,NDVI,2026_01_07_11_11_04,2026_01_07,1111
4,/home/ajai-krishna/work/Phenocam_d3/Phenocamda...,NDVI,2026_01_07_11_20_56,2026_01_07,1120
...,...,...,...,...,...
660,/home/ajai-krishna/work/Phenocam_d3/Phenocamda...,NDVI,2025_08_10_11_20_45,2025_08_10,1120
661,/home/ajai-krishna/work/Phenocam_d3/Phenocamda...,NDVI,2025_08_10_11_30_47,2025_08_10,1130
662,/home/ajai-krishna/work/Phenocam_d3/Phenocamda...,NDVI,2025_08_11_11_00_49,2025_08_11,1100
663,/home/ajai-krishna/work/Phenocam_d3/Phenocamda...,NDVI,2025_08_11_11_20_47,2025_08_11,1120


In [242]:
ndvi_img=np.array(Image.open(df_ndvi['file_path'][3]))
rows=cols=10
h, w = ndvi_img.shape[:2]
tile_h= h // rows
tile_w = w//rows
tiles = []
mean_ndvi = np.zeros((rows, cols))

for i in range(rows):
    for j in range(cols):
        tile = ndvi_img[
            i*tile_h:(i+1)*tile_h,
            j*tile_w:(j+1)*tile_w]
        tiles.append(tile)
        mean_ndvi[i,j]=np.nanmean(tile)

In [243]:
ndvi_img.shape

(1984, 2592, 3)

In [244]:
tiles[0].shape


(198, 259, 3)

In [245]:
dates = df1['date'].unique()
selected_fils = []
for date in dates:
    group = df1[df1['date'] == date]
    ir_times = group[group['image_type'] == 'ir']['time'].values
    rgb_times = group[group['image_type'] == 'rgb']['time'].values
    common_times = set(ir_times).intersection(set(rgb_times))
    time1 = common_times.pop() if common_times else None
    if time1:
        ir_file = group[(group['image_type'] == 'ir') & (group['time'] == time1)]['file_path'].values[0]
        rgb_file = group[(group['image_type'] == 'rgb') & (group['time'] == time1)]['file_path'].values[0]
        selected_fils.append({'date': date, 
                              'time': time1, 
                              'ir_file':ir_file, 
                              'rgb_file':rgb_file})


In [246]:
df=pd.DataFrame(selected_fils).sort_values(by=['date','time']).reset_index(drop=True)
df

,date,time,ir_file,rgb_file
0,2025_07_28,1344,/home/ajai-krishna/work/Phenocam_d3/Phenocamda...,/home/ajai-krishna/work/Phenocam_d3/Phenocamda...
1,2025_07_30,1330,/home/ajai-krishna/work/Phenocam_d3/Phenocamda...,/home/ajai-krishna/work/Phenocam_d3/Phenocamda...
2,2025_07_31,1330,/home/ajai-krishna/work/Phenocam_d3/Phenocamda...,/home/ajai-krishna/work/Phenocam_d3/Phenocamda...
3,2025_08_01,1110,/home/ajai-krishna/work/Phenocam_d3/Phenocamda...,/home/ajai-krishna/work/Phenocam_d3/Phenocamda...
4,2025_08_02,1110,/home/ajai-krishna/work/Phenocam_d3/Phenocamda...,/home/ajai-krishna/work/Phenocam_d3/Phenocamda...
...,...,...,...,...
165,2026_02_23,1131,/home/ajai-krishna/work/Phenocam_d3/Phenocamda...,/home/ajai-krishna/work/Phenocam_d3/Phenocamda...
166,2026_02_24,1131,/home/ajai-krishna/work/Phenocam_d3/Phenocamda...,/home/ajai-krishna/work/Phenocam_d3/Phenocamda...
167,2026_02_27,1131,/home/ajai-krishna/work/Phenocam_d3/Phenocamda...,/home/ajai-krishna/work/Phenocam_d3/Phenocamda...
168,2026_03_03,1132,/home/ajai-krishna/work/Phenocam_d3/Phenocamda...,/home/ajai-krishna/work/Phenocam_d3/Phenocamda...


In [247]:

def select_time(group):
    ir_time = group[group['image_type'] == 'ir']['time'].values
    rgb_df = group[group['image_type'] == 'rgb']['time'].values
    for time1 in ir_time:
        if time1 in rgb_df:
            return time1    

df1['temp']= df1.groupby(['date']).apply(select_time)

/tmp/ipykernel_13833/187874625.py:8: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df1['temp']= df1.groupby(['date']).apply(select_time)


In [248]:
df1['temp'].unique()
df1

,file_path,image_type,date_time,date,time,temp
0,/home/ajai-krishna/work/Phenocam_d3/Phenocamda...,other,2025_07_28_13_38_47,2025_07_28,1338,NaN
1,/home/ajai-krishna/work/Phenocam_d3/Phenocamda...,rgb,2025_07_28_13_38_47,2025_07_28,1338,NaN
2,/home/ajai-krishna/work/Phenocam_d3/Phenocamda...,other,2025_07_28_13_40_45,2025_07_28,1340,NaN
3,/home/ajai-krishna/work/Phenocam_d3/Phenocamda...,ir,2025_07_28_13_40_45,2025_07_28,1340,NaN
4,/home/ajai-krishna/work/Phenocam_d3/Phenocamda...,other,2025_07_28_13_42,2025_07_28,1342,NaN
...,...,...,...,...,...,...
2973,/home/ajai-krishna/work/Phenocam_d3/Phenocamda...,NDVI,2025_08_11_11_30_45,2025_08_11,1130,NaN
2974,/home/ajai-krishna/work/Phenocam_d3/Phenocamda...,other,2025_08_12_11_00,2025_08_12,1100,NaN
2975,/home/ajai-krishna/work/Phenocam_d3/Phenocamda...,other,2025_08_12_11_00,2025_08_12,1100,NaN
2976,/home/ajai-krishna/work/Phenocam_d3/Phenocamda...,other,2025_08_12_11_00,2025_08_12,1100,NaN


In [249]:
df['ndvi'] = df.apply(
    lambda row: ndvi_calculation(row['ir_file'], row['rgb_file']),
    axis=1)


In [250]:
save_ndvi_plot(df, plot_dir)

Saved NDVI plot to: /home/ajai-krishna/work/Phenocam_d3/Plots/ndvi_plot.png
Saved NDVI plot data to: /home/ajai-krishna/work/Phenocam_d3/Plots/ndvi_plot.json
